***01_cleaning_pipeline_demo.ipynb***  
***Developer Demonstration Notebook***  
This notebook demonstrates the cleaning pipeline process.
It shows:
 - how to run the pipeline
 - how to inspect audit artifacts
 - how to compare raw vs cleaned data
 - how to explore imputed rows
 - how to explore winsorization effects
 - how to inspect dropped rows

In [ ]:
import pandas as pd
import sys
import os
import json

# Ensure project root is on the path
project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.insert(0, project_root)

import logging

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s — %(levelname)s — %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S")

from src.customer_transaction_cleaner import DataCleaner
from src.customer_transaction_cleaner import log_startup
log_startup() 


In [ ]:
df = pd.read_csv("../data/raw/synthetic_transactions.csv")
print("Number of record:")
print(len(df))
df.head() 


In [ ]:
cleaner = DataCleaner()
df_clean, audit = cleaner.clean(df)

print(f"Original data memory usage: {df.memory_usage(deep=True).sum()} ")#
print(f"Cleaned data memory usage: {df_clean.memory_usage(deep=True).sum()}")
print(f"len df: {len(df)}, len clean_df: {len(df_clean)}")
print(df_clean)

In [ ]:
from IPython.display import Markdown, display
import json

f = json.load(open(audit.json_path))

audit_path_json = audit.json_path.replace("\\", "/")
display(Markdown(f"[Audit JSON]({audit_path_json})"))

audit_pretty = audit.json_pretty_path.replace("\\", "/")
display(Markdown(f"[Audit Pretty JSON]({audit_pretty})"))

audit_path_md = audit.md_path.replace("\\", "/")
display(Markdown(f"[Audit MarkDown]({audit_path_md})"))

summary_path = audit.process_summary_path.replace("\\", "/")
display(Markdown(f"[Process Summary]({summary_path})"))



In [ ]:
with open(audit.json_path, "r", encoding="utf-8") as f:
    audit_file = json.load(f)


In [ ]:
timestamp = [e for e in audit_file["events"] if e.get("column") == "timestamp"]
print(f"Column: timestamp")
print(json.dumps(timestamp, indent=4))

In [ ]:
row_dropped = [e for e in audit_file["events"] if e.get("dropped_rows") ]
print(f"Records dropped:")
print(json.dumps(row_dropped, indent=4))


In [ ]:

amount_audit = [e for e in audit_file["events"] if e.get("column") == "amount"]
print(f"all events for column amount")
print(json.dumps(amount_audit, indent=4))

In [ ]:
'''Imputation audit '''
imputation_events = [e for e in audit_file["events"] if e.get("event") == "ml_imputation"]
print(f"count of imputations {len(imputation_events)}")
print(json.dumps(imputation_events, indent=4))

In [ ]:
''' how to view the df changes from raw vs cleaned data'''
compare = [e for e in audit_file["events"] if e.get("event") == "compare_dataframes"]

print(f"compare_dataframes")
print(f"len df: {len(df)}, len clean_df: {len(df_clean)}")
print(json.dumps(compare, indent=4))


In [ ]:
'''winsorize events and rows'''
winsor_events = [e for e in audit_file["events"] if e.get("event") == "winsorization"]

winsor_rows = [
    idx
    for e in winsor_events
    for idx in e.get("row_indices", [])]

print(f"Number of winsorized rows: {len(winsor_rows)}")
print(json.dumps(winsor_events, indent=4))
print(json.dumps(winsor_rows, indent=4))

# print(json.dumps(audit_file, indent=4))




